## Goal

This tutorial walks through a compact but realistic fine-tuning workflow:

1. Build a transformer-based classifier.
2. Configure optimizer groups and weight decay.
3. Train with gradient clipping and validation tracking.
4. Save and reload checkpoints.

## Step 1: Build the Model


In [ ]:
#| eval: false
import torch
import torch.nn as nn

class EncoderClassifier(nn.Module):
    def __init__(self, vocab_size=200, embed_dim=64, heads=4, num_classes=3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attn = nn.MultiheadAttention(embed_dim, heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.GELU(),
            nn.Linear(128, embed_dim),
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        h = self.embed(x)
        a, _ = self.attn(h, h, h)
        h = self.norm1(h + a)
        h = self.norm2(h + self.ffn(h))
        cls = h[:, 0, :]
        return self.head(cls)


## Step 2: Optimizer Parameter Groups

A common best practice is to apply weight decay to most parameters, but not to biases and layer norm weights.


In [ ]:
#| eval: false
def build_optimizer(model, lr=2e-4, wd=0.01):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.endswith("bias") or "norm" in n.lower():
            no_decay.append(p)
        else:
            decay.append(p)

    groups = [
        {"params": decay, "weight_decay": wd},
        {"params": no_decay, "weight_decay": 0.0},
    ]
    return torch.optim.AdamW(groups, lr=lr)


## Step 3: Training Loop


In [ ]:
#| eval: false
from torch.utils.data import TensorDataset, DataLoader

def make_data(n=2000, seq_len=20, vocab_size=200):
    X, y = [], []
    for _ in range(n):
        label = torch.randint(0, 3, (1,)).item()
        seq = torch.randint(4, vocab_size, (seq_len,))
        seq[0] = label + 1
        X.append(seq)
        y.append(label)
    return TensorDataset(torch.stack(X), torch.tensor(y))

def run_training(model, train_dl, val_dl, epochs=8):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    opt = build_optimizer(model)
    loss_fn = nn.CrossEntropyLoss()

    for ep in range(1, epochs + 1):
        model.train()
        tr_correct = tr_total = 0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr_correct += (logits.argmax(-1) == yb).sum().item()
            tr_total += len(yb)

        model.eval()
        va_correct = va_total = 0
        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb).argmax(-1)
                va_correct += (pred == yb).sum().item()
                va_total += len(yb)

        print(f"Epoch {ep}: train_acc={tr_correct/tr_total:.3f}, val_acc={va_correct/va_total:.3f}")

    return model

ds = make_data()
train_ds, val_ds = torch.utils.data.random_split(ds, [1600, 400])
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=64)

model = EncoderClassifier()
model = run_training(model, train_dl, val_dl, epochs=8)


## Step 4: Checkpointing


In [ ]:
#| eval: false
ckpt_path = "m04_t1_classifier.pt"

torch.save({"state_dict": model.state_dict()}, ckpt_path)
print(f"Saved checkpoint to {ckpt_path}")

new_model = EncoderClassifier()
payload = torch.load(ckpt_path, map_location="cpu")
new_model.load_state_dict(payload["state_dict"])
print("Checkpoint reloaded successfully")


## Exercises

1. Freeze the embedding layer and compare validation accuracy.
2. Add a learning-rate scheduler with warmup.
3. Replace GELU with ReLU in the FFN and compare outcomes.
4. Add early stopping based on validation accuracy.
5. Export predictions for 100 validation samples and inspect error cases.
